# 1. Objective
  The Objective of the notebook is to perform data quality checks beforehand and ensure all the basic data requirements are met before proceeding with the later tasks. If there are any significant data issues identified, then the notebook returns an error message
  Following are the data checks that are performed
   - Check if all the required columns as mentioned in the config are present in the harmonized data
   - Check if the data types of the columns are as per expectation
   - Check if Promo related columns are within a range (0-100)
   - Check if any REGIONNAME x F_CODE is missed compoared to the previous iteration
   - Check if any new EXTRA_WORDS got classified to OTHERS. - Potential indicator for a new promo
   - Check for data drift and report

# 2. Imports

In [ ]:
# Import python packages

import pandas as pd
import numpy as np
import os

# MUST be set before any numba / evidently import
os.environ["NUMBA_DISABLE_JIT"] = "1"
os.environ["NUMBA_CACHE_DIR"] = "/tmp"

from evidently.report import Report
from evidently.metric_preset import DataDriftPreset
from evidently.options import DataDriftOptions
from snowflake.snowpark import functions as F

from snowflake.snowpark import types as T
from snowflake.snowpark import Session
# df = pd.read_excel('Harmonized Data - Validation.xlsx',sheet_name = 'Sample V2')
# print(df.shape)
# df.head()
# We can also use Snowpark for our analyses!
from snowflake.snowpark.context import get_active_session
session = get_active_session()

In [ ]:
print("✅ Snowpark Session Initialized Successfully!")
print("Current Database:", session.get_current_database())
print("Current Schema:", session.get_current_schema())

# 3. Setup environment

## 3.1. Load Config

In [ ]:
import yaml
with open("config_new_PROD.yaml") as file:
    app_config = yaml.safe_load(file)

## 3.2. Update Output Database, Schema , table

In [ ]:
output_database = app_config["general_inputs"]["output_database"]
output_schema = app_config["general_inputs"]["output_schema"]
print(output_database, output_schema)

In [ ]:
session.use_database(output_database)
session.use_schema(output_schema)
data_drift_output_table_name = "PROD_DATA_DRIFT_SUMMARY"
data_qc_output_table_name = "PROD_DATA_QC_SUMMARY"
rep_schema_data_drift_output_table_name = "DATA_DRIFT_SUMMARY"
rep_schema_data_qc_output_table_name = "DATA_QC_SUMMARY"

In [ ]:
# Example check (optional)
print("✅ Snowpark Session Initialized Successfully!")
print("Current Database:", session.get_current_database())
print("Current Schema:", session.get_current_schema())

## 3.3. Export data API

In [ ]:
def snowpark_type_to_sql(datatype):
    if isinstance(datatype, T.StringType):
        return "VARCHAR"
    if isinstance(datatype, T.IntegerType):
        return "INTEGER"
    if isinstance(datatype, T.LongType):
        return "BIGINT"
    if isinstance(datatype, T.ShortType):
        return "SMALLINT"
    if isinstance(datatype, T.ByteType):
        return "BYTEINT"
    if isinstance(datatype, T.BooleanType):
        return "BOOLEAN"
    if isinstance(datatype, T.FloatType):
        return "FLOAT"
    if isinstance(datatype, T.DoubleType):
        return "DOUBLE"
    if isinstance(datatype, T.DecimalType):
        return f"NUMBER({datatype.precision},{datatype.scale})"
    if isinstance(datatype, T.DateType):
        return "DATE"
    if isinstance(datatype, T.TimestampType):
        return "TIMESTAMP_NTZ"
    if isinstance(datatype, T.TimestampTimeZoneType):
        return "TIMESTAMP_TZ"
    if isinstance(datatype, T.VariantType):
        return "VARIANT"

    # Fallback (safe but explicit)
    return "VARCHAR"

In [ ]:
def table_exists(session, table_fqn):
    try:
        session.table(table_fqn).limit(1).collect()
        return True
    except Exception:
        return False

In [ ]:
def evolve_schema_and_append_with_TS(
    session: Session,
    src_df: str,
    target_table_name: str,
):
    """
    Schema-drift tolerant append:
      - Adds new columns from source to target
      - Appends data with column alignment
    """

    # 0️⃣ Create target if missing
    if not table_exists(session, target_table_name):
        (
            src_df
            .limit(0)
            .write
            .mode("overwrite")
            .save_as_table(target_table_name)
        )
    #src_df = session.table(source_table)
    tgt_df = session.table(target_table_name)
    src_df = src_df.with_column("LOAD_TS", F.current_timestamp())
    src_df = src_df.with_column("EXECUTION_YEAR_MONTH",F.date_format(F.current_date(), "YYYY-MM"))
    src_row_count = src_df.count()
    src_col_count = len(src_df.columns)

    print("Source DF shape ->",src_row_count,",",src_col_count)

    tgt_row_count = tgt_df.count()
    tgt_col_count = len(tgt_df.columns)

    print("Target table shape ->",tgt_row_count,",",tgt_col_count)

    src_schema = {f.name.upper(): f.datatype for f in src_df.schema.fields}
    tgt_schema = {f.name.upper(): f.datatype for f in tgt_df.schema.fields}

    # 1️⃣ Add missing columns to target
    new_columns = src_schema.keys() - tgt_schema.keys()
    if len(new_columns)>0:
        print("New Columns in Source dataframe found. Target table will be altered")
    else:
        print("No New columns found in source dataframe")
    for col_name in new_columns:
        datatype = snowpark_type_to_sql(src_schema[col_name])
        print(col_name)
        print(datatype)
        ddl = f"""
            ALTER TABLE {target_table_name}
            ADD COLUMN IF NOT EXISTS "{col_name}" {datatype}
        """
        print(ddl)
        ddl_op = session.sql(ddl)
        print(ddl_op.collect())
    
    # Refresh target after DDL
    tgt_df = session.table(target_table_name)

    # Align columns for insert
    common_cols = [
        F.col(c)
        for c in tgt_df.schema.names
        if c.upper() in src_schema
    ]
    
    (
        src_df
        .select(common_cols)
        .write
        .mode("append")
        .save_as_table(target_table_name, column_order = "name")
    )
    print("Appended ",src_row_count," rows of data to Target table with latest TS successfully")

# 4. QC

In [ ]:
## Read Current month data
test_df = session.table("PROD_TABLES.PROD_ADS_STABLE")
print("All data ->", test_df.count())
latest_ts = test_df.select(F.max("LOAD_TS")).collect()[0][0]
second_max_df = test_df.select("LOAD_TS").distinct().sort(F.col("LOAD_TS").desc()).limit(2)
second_max_ts = second_max_df[0][1]
print(latest_ts)
df = test_df.filter(F.col("LOAD_TS") == F.lit(latest_ts))
print("Latest data ->", df.count())

In [ ]:
# Read previous month data
test_df = test_df.filter(F.col("LOAD_TS")!=F.lit(latest_ts))
print("All data ->", test_df.count())
latest_ts = test_df.select(F.max("LOAD_TS")).collect()[0][0]
print(latest_ts)
df_prev_month = test_df.filter(F.col("LOAD_TS") == F.lit(latest_ts))
print("Latest data ->", df_prev_month.count())
df_prev_month_pd = df_prev_month.to_pandas()

In [ ]:
df_pd = df.to_pandas()

In [ ]:
df_pd

## 4.1. Check if all the required columns are present

In [ ]:
dv = app_config["general_inputs"]["dependent_variable"]
ds = app_config["general_inputs"]["date_var"]
modeling_granularity_conf = app_config["general_inputs"]["modeling_granularity"]
missing_value_cols = app_config["data_processing"]["missing_value_treatment"]["Mean"]["cols"]+app_config["data_processing"]["missing_value_treatment"]["Median"]["cols"] + app_config["data_processing"]["missing_value_treatment"]["Scalar"]["cols"] + app_config["data_processing"]["missing_value_treatment"]["Rolling_Mean"]["cols"]+app_config["data_processing"]["missing_value_treatment"]["Rolling_Median"]["cols"]+app_config["data_processing"]["missing_value_treatment"]["Forward_fill"]["cols"]+app_config["data_processing"]["missing_value_treatment"]["Backward_fill"]["cols"]+app_config["data_processing"]["missing_value_treatment"]["Linear_Interpolation"]["cols"]+app_config["data_processing"]["missing_value_treatment"]["Spline_Interpolation"]["cols"]+app_config["data_processing"]["missing_value_treatment"]["Mode"]["cols"]+app_config["data_processing"]["missing_value_treatment"]["Mean_Across_Years"]["cols"]+app_config["data_processing"]["missing_value_treatment"]["Zero"]["cols"]


final_req_cols = [dv,ds] + modeling_granularity_conf + missing_value_cols
print(len(final_req_cols))

In [ ]:
# Checking whether required columns are present in the given data

def check_required_columns(df, required_columns):
    missing = set(required_columns) - set(df.columns)
    if missing:
        qc_output = "1 or more required columns are missing in the Harmonized data"
        raise ValueError(f"Missing required columns: {missing}")
    else:
        qc_output = "All good. Required columns are present in the Harmonized data"
    return qc_output

#required_cols = ["F_CODE", "REGIONNAME", "WINBACK_WEIGHTED_MEAN"]
qc_output = check_required_columns(df, final_req_cols)
print(qc_output)

In [ ]:
qc_df1 = pd.DataFrame({"QC_NUMBER":[1],"QC_DESCRIPTION":["Check if all the required columns are present in Harmonized data"],"QC_RESULT":[qc_output]})
qc_df1

## 4.2. Check if all the modeling granularity columns are of expected data type

In [ ]:
# checking whether data types are matching.

def validate_dtypes(actual_dtypes: dict, required_dtypes: dict):
    """
    actual_dtypes:  {"col1": "int", "col2": "string"}
    required_dtypes:{"col1": "int", "col2": "float"}
    """
    mismatches = {}

    for col, req_type in required_dtypes.items():
        actual_type = actual_dtypes.get(col)

        if actual_type is None:
            mismatches[col] = f"Missing column (expected {req_type})"
        elif actual_type.lower() != req_type.lower():
            mismatches[col] = f"Expected {req_type}, got {actual_type}"

    return mismatches

actual = {"F_CODE": str(df_pd['F_CODE'].dtype), "REGIONNAME": str(df_pd['REGIONNAME'].dtype)}
required = {"F_CODE": "object", "REGIONNAME": "object"}

errors = validate_dtypes(actual, required)

if errors:
    #print("Data type mismatches:")
    qc_output = "Unexpected data type found"
    for k, v in errors.items():
        print(f"{k}: {v}")
else:
    qc_output = "All good. Modeling granularity columns data types match the expected type"
    #print("All data types match")
print(qc_output)

In [ ]:
qc_df2 = pd.DataFrame({"QC_NUMBER":[2],"QC_DESCRIPTION":["Check if all the modeling granularity columns are of expected data type"],"QC_RESULT":[qc_output]})
qc_df2

In [ ]:
def split_recent_weeks(df,granularity_cols,week_col,n_weeks):
    
    df = df.copy()

    # rank weeks per granularity (latest = 1)
    df["_week_rank"] = (
        df
        .sort_values(week_col, ascending=False)
        .groupby(granularity_cols)[week_col]
        .rank(method="dense", ascending=False)
    )

    recent_df = df[df["_week_rank"] <= n_weeks]
    history_df = df[df["_week_rank"] > n_weeks]

    return recent_df.drop(columns="_week_rank"), history_df.drop(columns="_week_rank")

recent_df, history_df = split_recent_weeks(df_pd,granularity_cols=["F_CODE","REGIONNAME"],
                                           week_col="START_OF_WEEK",n_weeks=2)

## 4.3. Check if Studio weights sum is 1 for all Studio groups

In [ ]:
test_df = session.table("PROD_TABLES.PROD_STUDIO_IMPORTANCE")
print("All data ->", test_df.count())
latest_ts = test_df.select(F.max("LOAD_TS")).collect()[0][0]
studio_weights = test_df.filter(F.col("LOAD_TS") == F.lit(latest_ts))
print("Latest data ->", studio_weights.count())

In [ ]:
studio_weights

In [ ]:
agg_df = studio_weights.group_by(["F_CODE"]).agg(F.sum("STUDIO_IMPORTANCE").alias("STUDIO_SUM"))
agg_df = agg_df.with_column("EXPECTED_SUM",F.lit(1))
agg_df = agg_df.with_column("PERC_DIFF", F.abs(((F.col("EXPECTED_SUM") - F.col("STUDIO_SUM"))/F.col("EXPECTED_SUM"))*100))
rows_with_issues = agg_df.filter(F.col("PERC_DIFF")>0.1).count()
if rows_with_issues>0:
    qc_output = "There are F_CODEs where the individual studio weights sum is not 1"
else:
    qc_output = "All good. For all FCODEs, the individual studio weights sum up to 1"
print(qc_output)
agg_df

In [ ]:
qc_df3 = pd.DataFrame({"QC_NUMBER":[3],"QC_DESCRIPTION":["Check if Studio weights sum is 1 for all Studio groups"],"QC_RESULT":[qc_output]})
qc_df3

## 4.4. Check if the promo attributes are within the expected range

In [ ]:
def row_level_violations(df):
    cols = [
        c for c in df.columns
        if c.endswith("_WEIGHTED_MEAN") or c.endswith("_NORMALIZED") or c.endswith("_MEAN")
    ]

    violations = (
        df[cols]
        .apply(lambda s: s.where(~s.between(0, 100)))
        .stack()
        .reset_index()
    )

    violations.columns = ["row_index", "column_name", "invalid_value"]
    return violations

df2 = row_level_violations(df_pd)
if len(df2)>0:
    qc_output = "There are "+len(df2)+" rows where promo data is beyond the bounds -> 0-100"
else:
    qc_output = "All good. Promo data is within the bounds -> 0-100"
print(qc_output)
print(df2)

In [ ]:
qc_df4 = pd.DataFrame({"QC_NUMBER":[4],"QC_DESCRIPTION":["Check if the promo attributes are within the expected range"],"QC_RESULT":[qc_output]})
qc_df4

## 4.5. Check if any REGIONNAME x F_CODE is missing

In [ ]:
prev_iter_fcodes = df_prev_month_pd.groupby(["REGIONNAME","F_CODE"]).agg({"START_OF_WEEK":"count"})
prev_iter_fcodes = prev_iter_fcodes.rename(columns = {"START_OF_WEEK":"NO_OF_WEEKS_PREV_ITER"})
curr_iter_fcodes = df_pd.groupby(["REGIONNAME","F_CODE"]).agg({"START_OF_WEEK":"count"})
curr_iter_fcodes = curr_iter_fcodes.rename(columns = {"START_OF_WEEK":"NO_OF_WEEKS_CURR_ITER"})
prev_iter_fcodes = prev_iter_fcodes.merge(curr_iter_fcodes, on = ["REGIONNAME","F_CODE"], how = "left")
issues = prev_iter_fcodes["NO_OF_WEEKS_CURR_ITER"].isnull().sum()
if issues>0:
    qc_output = "Missing FCODEs. There are "+ str(issues)+ " missing REGIONNAME x F_CODEs in the current iteration"
else:
    qc_output = "All good. All the F_CODEs in the last iteration are present in the current iteration"
print(qc_output)

In [ ]:
qc_df5 = pd.DataFrame({"QC_NUMBER":[5],"QC_DESCRIPTION":["Check if any REGIONNAME x F_CODE is missing"],"QC_RESULT":[qc_output]})
qc_df5

In [ ]:
final_qc_df = pd.concat([qc_df1,qc_df2,qc_df3,qc_df4,qc_df5], axis = 0)
final_qc_df.reset_index(drop = True)

In [ ]:
qc_df_spk = session.create_dataframe(final_qc_df.reset_index(drop = True))

In [ ]:
evolve_schema_and_append_with_TS(session, src_df = qc_df_spk, target_table_name = data_qc_output_table_name)

# 5. Data Drift Metrics

In [ ]:
df_pd["START_OF_WEEK"] = pd.to_datetime(df_pd["START_OF_WEEK"])

# Returns historical and recent_4_weeks as dataframe as dictionary
def split_data_by_region_fcode(df, reference_date):
   
    output = {}

    for comb, g in df.groupby(["REGIONNAME","F_CODE"]):
        g = g.sort_values("START_OF_WEEK")
        region = comb[0]
        f_code = comb[1]

        current_df = g[g["START_OF_WEEK"] >= reference_date]
        reference_df = g[g["START_OF_WEEK"] < reference_date]
        print(current_df.shape)
        print(reference_df.shape)
        if len(current_df) == 0 or len(reference_df) == 0:
            continue  # skip insufficient data

        output[region+"_"+f_code] = {
            "reference": reference_df.drop(columns=["REGIONNAME","F_CODE", "START_OF_WEEK"]),
            "current": current_df.drop(columns=["REGIONNAME","F_CODE", "START_OF_WEEK"])
        }

    return output

In [ ]:
# To skip Columns with Insufficient Data

def prepare_for_drift(reference_df, current_df, min_non_null=1):
    valid_cols = [
        col for col in reference_df.columns
        if (
            reference_df[col].notna().sum() >= min_non_null and
            current_df[col].notna().sum() >= min_non_null
        )
    ]
    return reference_df[valid_cols], current_df[valid_cols]


In [ ]:
# Run Evidently Data Drift for Each F_CODE 
df_prev_month_pd["START_OF_WEEK"] = pd.to_datetime(df_prev_month_pd["START_OF_WEEK"])
max_date_last_month = df_prev_month_pd["START_OF_WEEK"].max()
max_date_last_month = pd.to_datetime("20-11-2025")

In [ ]:
fcode_data = split_data_by_region_fcode(df_pd, max_date_last_month)

In [ ]:
drift_summary = {}

for comb, data in fcode_data.items():
    
    report = Report(metrics=[DataDriftPreset()])

    ref_df, curr_df = prepare_for_drift(data["reference"],data["current"],
                                        min_non_null=2)

    if curr_df.empty or curr_df.shape[1] == 0:
        continue
    
    report.run(
        reference_data=ref_df,
        current_data=curr_df
    )

    drift_summary[comb] = report.as_dict()

# Extract Column-Level Drift

column_drift_rows = []

for comb, report_dict in drift_summary.items():
    region, f_code = comb.split("_")
    columns = report_dict["metrics"][1]["result"]["drift_by_columns"]

    for col_name, col_metrics in columns.items():
        column_drift_rows.append({
            "REGIONNAME":region,
            "F_CODE": f_code,
            "column": col_name,
            "drift_detected": col_metrics["drift_detected"],
            "drift_score": col_metrics.get("drift_score"),
            "stattest": col_metrics.get("stattest_name"),
        })

column_drift_df = pd.DataFrame(column_drift_rows)


column_drift_df

In [ ]:
column_drift_df_spk = session.create_dataframe(column_drift_df)

In [ ]:
evolve_schema_and_append_with_TS(session, src_df = column_drift_df_spk, target_table_name = data_drift_output_table_name)

# 6. Export reports to ML_MONITORING schema

In [ ]:
session.use_schema("ML_MONITORING")

In [ ]:
ddl = f"""
            CREATE OR REPLACE TABLE {rep_schema_data_qc_output_table_name} AS SELECT * FROM {output_schema}.{data_qc_output_table_name}
"""
print(ddl)
ddl_op = session.sql(ddl)
print(ddl_op.collect())

In [ ]:
ddl = f"""
            CREATE OR REPLACE TABLE {rep_schema_data_drift_output_table_name} AS SELECT * FROM {output_schema}.{data_drift_output_table_name}
"""
print(ddl)
ddl_op = session.sql(ddl)
print(ddl_op.collect())